[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/tracing/agentic_rag_tracing.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>
<h1 align="center">追踪智能体 RAG 应用</h1>

本教程将：

- 构建智能体 RAG 应用
- 使用 Phoenix 为应用添加埋点并追踪
- 在 Phoenix 中检查追踪数据

## 背景

智能体 RAG（检索增强生成）将传统 RAG 系统与能够决策和行动的自主智能体结合。传统 RAG 直接检索相关上下文并生成回答，智能体 RAG 则进一步支持：

- 将复杂查询拆解为子任务
- 选择适当的工具和操作完成任务
- 对多来源信息进行推理和综合
- 判断哪些信息相关

例如，智能体 RAG 系统可能按以下流程处理问题：
1. 分析用户问题，确定所需信息
2. 按需查询多个数据源或工具
3. 整合检索信息并推理
4. 生成全面的回答

本教程使用 LlamaIndex 的 ReAct 智能体框架，结合向量和 SQL 查询工具构建智能体 RAG 系统。

开始吧！

ℹ️ 本 Notebook 需要 OpenAI API 密钥。

## 导入依赖

In [ ]:
!pip install -q llama-index openai arize-phoenix-otel openinference-instrumentation-llama-index llama-index-vector-stores-chroma chromadb sqlalchemy

In [ ]:
import os
from getpass import getpass

import chromadb
import chromadb.utils.embedding_functions as embedding_functions
from llama_index.core import VectorStoreIndex
from llama_index.core.agent import ReActAgent
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.chroma import ChromaVectorStore
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from phoenix.otel import register

## 添加密钥

### OpenAI API 密钥

In [ ]:
if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 请输入 OpenAI API 密钥： ")

os.environ["OPENAI_API_KEY"] = openai_api_key

### Phoenix 密钥

可以在本地运行 Phoenix，或通过 phoenix.arize.com 免费使用。

In [ ]:
if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 请输入 Phoenix API 密钥： ")

os.environ["PHOENIX_CLIENT_HEADERS"] = f"api_key={phoenix_api_key}"

## 为智能体添加埋点

将 Notebook 连接到 Phoenix，即可追踪智能体的操作并查看追踪数据。

Phoenix 自动埋点会追踪所有 LlamaIndex 调用，无需额外添加埋点。

In [ ]:
# 配置 Phoenix 追踪器
tracer_provider = register(
    project_name="agentic-rag-demo",
    endpoint="https://app.phoenix.arize.com/v1/traces",  # 若在本地运行 Phoenix，请修改此端点
)

# 完成自动插桩
LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider)

## 使用 Chroma 构建查询引擎工具

现在创建智能体回答问题时要使用的两个数据库。使用向量数据库 Chroma 存储公司政策和员工信息。

### 公司政策数据库

In [ ]:
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"], model_name="text-embedding-3-small"
)

In [ ]:
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection(
    "agentic-rag-demo-company-policies", embedding_function=openai_ef
)

chroma_collection.add(
    ids=["1", "2", "3"],
    documents=[
        "The travel policy is: Employees must book travel through the company portal. Economy class flights and standard hotel rooms are covered. Meals during travel are reimbursed up to $75/day. All expenses require receipts.",
        "The pto policy is: Full-time employees receive 20 days of paid time off per year, accrued monthly. PTO requests must be submitted at least 2 weeks in advance through the HR portal. Unused PTO can carry over up to 5 days into the next year.",
        "The dress code is: Business casual attire is required in the office. This includes collared shirts, slacks or knee-length skirts, and closed-toe shoes. Jeans are permitted on Fridays. No athletic wear or overly casual clothing.",
    ],
)

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

chroma_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
chroma_engine_policy = chroma_index.as_query_engine(similarity_top_k=1)

In [ ]:
chroma_collection.peek()

In [ ]:
chroma_engine_policy.query("What is the travel policy?")

### 员工数据库

In [ ]:
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection(
    "agentic-rag-demo-company-employees", embedding_function=openai_ef
)

chroma_collection.add(
    ids=["1", "2", "3"],
    documents=[
        "John Smith is a Software Engineer in the Engineering department who started on 2023-01-15",
        "Sarah Johnson is a Marketing Manager in the Marketing department who started on 2022-08-01",
        "Michael Williams is a Sales Director in the Sales department who started on 2021-03-22",
    ],
)

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

chroma_index = VectorStoreIndex.from_vector_store(vector_store=vector_store)
chroma_engine_employees = chroma_index.as_query_engine(similarity_top_k=1)

In [ ]:
chroma_engine_employees.query("What is the name of the Sales Director?")

### 添加为工具

LlamaIndex 的 ReAct 智能体框架支持向智能体添加工具。下面添加用于回答问题的两个工具。

In [ ]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=chroma_engine_employees,
        metadata=ToolMetadata(
            name="ChromaEmployees",
            description=(
                "从关系数据库提供员工部门、入职日期和姓名信息。"
                "使用详细的纯文本问题作为工具输入。"
            ),
        ),
    ),
    QueryEngineTool(
        query_engine=chroma_engine_policy,
        metadata=ToolMetadata(
            name="ChromaPolicy",
            description=(
                "提供公司政策与流程信息，可用于获取更详细的公司政策内容。"
                "使用针对具体政策的详细纯文本陈述作为工具输入。"
            ),
        ),
    ),
]

## 创建 ReAct 智能体

LlamaIndex 的 ReAct 智能体框架支持创建使用工具回答问题的智能体。下面创建智能体，使其能够使用刚才创建的两个工具。

In [ ]:
CONTEXT = """
你是回答公司员工与政策问题的聊天机器人。
你可以访问保存公司员工与部门信息的 Chroma 数据库。
你也可以访问保存公司政策的 Chroma 数据库。请利用已提供的上下文回答
问题。确保掌握回答问题所需的全部上下文；若缺少信息，请检查是否有
其他工具可以帮助回答。若仍无法回答，请向用户索取更多信息，并
说明无法回答并致歉。
"""

In [ ]:
llm = OpenAI(model="gpt-3.5-turbo")

agent = ReActAgent.from_tools(query_engine_tools, llm=llm, verbose=True, context=CONTEXT)

## 试运行！

In [ ]:
response = agent.chat("What department is Sarah in?")
print(str(response))

In [ ]:
response = agent.chat("What is the pto policy for the ML Solutions team?")
print(str(response))